### Data Ingetion

In [1]:
### document datastructure

from langchain_core.documents import Document

In [2]:
doc = Document(
    page_content= "this is the main text content i am using to create RAG",
    metadata ={
    "source" : "example.txt",
    "pages" : 1,
    "author" : "Mahesh Kakade",
    "date_created" : "2024-06-05"
    }
)
doc

Document(metadata={'source': 'example.txt', 'pages': 1, 'author': 'Mahesh Kakade', 'date_created': '2024-06-05'}, page_content='this is the main text content i am using to create RAG')

In [3]:
## Create document directory
import os
os.makedirs('../data/documents', exist_ok=True)


In [4]:
import os

sample_txt = {
    "../data/documents/python_intro.txt": """Python is a high-level,
      easy-to-learn programming language with simple syntax 
      and a large standard library that supports tasks like web development,
      data science, and automation.

Key features:
- Interpreted
- Cross-platform
- Widely used in AI/ML, web development, and automation
""",

    "../data/documents/machine_learning.txt": """Machine Learning (AI/ML) is a field of
    computer science where systems learn from data and make predictions or decisions without being 
    explicitly programmed.

Key features:
- Used in prediction tasks
- Classification
- Recommendation systems
- Image recognition
"""
}

for filepath, content in sample_txt.items():
    os.makedirs(os.path.dirname(filepath), exist_ok=True)
    with open(filepath, 'w', encoding="utf-8") as f:
        f.write(content)

print("Sample text files created!")

Sample text files created!


In [5]:
###Textloader
from langchain_community.document_loaders import TextLoader

loader = TextLoader(
    r"c:\Users\Mahesh\rag_project\data\documents\python_intro.txt",
    encoding="utf-8"
)

docs = loader.load()
print(docs)


C:\Users\Mahesh\AppData\Local\Temp\ipykernel_8264\2148723557.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import TextLoader
c:\Users\Mahesh\rag_project\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


[Document(metadata={'source': 'c:\\Users\\Mahesh\\rag_project\\venv\\data\\text_file\\python_intro.txt'}, page_content='Python is a high-level,\n      easy-to-learn programming language with simple syntax \n      and a large standard library that supports tasks like web development,\n      data science, and automation.\n\nKey features:\n- Interpreted\n- Cross-platform\n- Widely used in AI/ML, web development, and automation\n')]


In [6]:
## Directory loader
from langchain_community.document_loaders import DirectoryLoader, TextLoader

dir_loader = DirectoryLoader(
    r"c:\Users\Mahesh\rag_project\data\documents",
    glob="*.txt",
    loader_cls=TextLoader,
    loader_kwargs={"encoding": "utf-8"},
    show_progress=True
)

documents = dir_loader.load()
documents

100%|â–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆ| 2/2 [00:00<00:00, 995.92it/s]


[Document(metadata={'source': 'c:\\Users\\Mahesh\\rag_project\\venv\\data\\text_file\\machine_learning.txt'}, page_content='Machine Learning (AI/ML) is a field of\n    computer science where systems learn from data and make predictions or decisions without being \n    explicitly programmed.\n\nKey features:\n- Used in prediction tasks\n- Classification\n- Recommendation systems\n- Image recognition\n'),
 Document(metadata={'source': 'c:\\Users\\Mahesh\\rag_project\\venv\\data\\text_file\\python_intro.txt'}, page_content='Python is a high-level,\n      easy-to-learn programming language with simple syntax \n      and a large standard library that supports tasks like web development,\n      data science, and automation.\n\nKey features:\n- Interpreted\n- Cross-platform\n- Widely used in AI/ML, web development, and automation\n')]

In [7]:
from langchain_community.document_loaders import DirectoryLoader, PyMuPDFLoader

dir_loader = DirectoryLoader(
    r"C:\Users\Mahesh\rag_project\data\documents",
    glob="**/*.pdf",
    loader_cls=PyMuPDFLoader,  # loader class to use
    show_progress=False
)

pdf_documents = dir_loader.load()
pdf_documents


[Document(metadata={'producer': 'MicrosoftÂ® Word 2024', 'creator': 'MicrosoftÂ® Word 2024', 'creationdate': '2026-04-30T11:25:15+05:30', 'source': 'C:\\Users\\Mahesh\\rag_project\\venv\\data\\text_file\\Internship report.pdf', 'file_path': 'C:\\Users\\Mahesh\\rag_project\\venv\\data\\text_file\\Internship report.pdf', 'total_pages': 25, 'format': 'PDF 1.7', 'title': 'Format for PBS', 'author': 'Nielsh J Uke', 'subject': 'TE IT', 'keywords': '', 'moddate': '2026-04-30T11:25:15+05:30', 'trapped': '', 'modDate': "D:20260430112515+05'30'", 'creationDate': "D:20260430112515+05'30'", 'page': 0}, page_content='An Internship Report \n \nSubmitted to the Savitribai Phule \nPune University \n \n \nIn partial Fulfillment for the award of the degree of Bachelor of \nEngineering \nIn Information Technology \nBy \n \nMahesh Vijay Kakade \n \n \n \n \n \nUnder the Guidance of \n \nProf. Ashwini Taksal \n \n \nDepartment of Information Technology \nDhole Patil College of Engineering \nKharadi, Pune, 

In [8]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)

chunks = text_splitter.split_documents(pdf_documents)

print("Total chunks:", len(chunks))

Total chunks: 61


### Embedding and vectorstoreDB

In [9]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity

In [10]:
class  EmbeddingManager:
    """Handles document embedding generation using SentenceTransformer"""

    def __init__(self,model_name: str = "all-MiniLM-L6-v2"):
        """Initialize the embedding manager
        Args:
            model_name: HuggingFace model name for sentence embeddings
        """
        self.model_name = model_name
        self.model = None
        self._load_model()

    def _load_model(self):
        """Load the sentencetranceformer model"""
        try:
            print(f"Loading embedding model: {self.model_name}")
            self.model = SentenceTransformer(self.model_name)
            print(f"Model loaded successfully. Embedding dimension: {self.model.get_sentence_embedding_dimension()} ")
        except Exception as e:
            print(f"Error Loading model {self.model_name}:{e}")
            raise

    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        """Generate embeddings for a list of texts
        """
        if self.model is None:
            raise ValueError("Model not loaded")

        print(f"Generating embeddings for {len(texts)} texts...")

        embeddings = self.model.encode(
            texts,
            show_progress_bar=True
        )

        print(f"Generated embeddings with shape: {embeddings.shape}")

        return embeddings
embedding_manager = EmbeddingManager()

Loading embedding model: all-MiniLM-L6-v2


Loading weights: 100%|â–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆ| 103/103 [00:00<00:00, 3545.98it/s]


Model loaded successfully. Embedding dimension: 384 


C:\Users\Mahesh\AppData\Local\Temp\ipykernel_8264\14846132.py:18: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print(f"Model loaded successfully. Embedding dimension: {self.model.get_sentence_embedding_dimension()} ")


## vectore store

In [11]:
import chromadb
import numpy as np
import hashlib
import os
from typing import List, Any


class Vector_Store:
    """Manage document embeddings in a persistent ChromaDB vector store."""

    def __init__(
        self,
        collection_name: str = "pdf_documents",
        persist_directory: str = "../data/vector_store"
    ):
        self.collection_name = collection_name
        self.persistent_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_store()

    def _initialize_store(self):
        """Initialize ChromaDB client and collection."""
        try:
            os.makedirs(self.persistent_directory, exist_ok=True)

            self.client = chromadb.PersistentClient(
                path=self.persistent_directory
            )

            self.collection = self.client.get_or_create_collection(
                name=self.collection_name,
                metadata={
                    "description": "PDF document embeddings for RAG"
                }
            )

            print(
                f"Vector store initialized. "
                f"Collection: {self.collection_name}"
            )
            print(
                f"Existing documents in collection: "
                f"{self.collection.count()}"
            )

        except Exception as e:
            print(f"Error initializing vector store: {e}")
            raise

    def add_documents(
        self,
        documents: List[Any],
        embeddings: np.ndarray
    ):
        """Add documents without creating duplicate records."""

        if len(documents) != len(embeddings):
            raise ValueError(
                "Number of documents must match number of embeddings"
            )

        ids = []
        metadatas = []
        documents_txt = []
        embeddings_list = []

        for i, (doc, embedding) in enumerate(
            zip(documents, embeddings)
        ):
            source = str(doc.metadata.get("source", "unknown"))
            page = str(doc.metadata.get("page", "unknown"))

            content_hash = hashlib.sha256(
                doc.page_content.encode("utf-8")
            ).hexdigest()[:16]

            doc_id = f"{source}|{page}|{content_hash}"
            ids.append(doc_id)

            metadata = dict(doc.metadata)
            metadata["doc_index"] = i
            metadata["content_length"] = len(doc.page_content)
    
            metadatas.append(metadata)
            documents_txt.append(doc.page_content)
            embeddings_list.append(embedding.tolist())

        try:
            existing = self.collection.get(ids=ids, include=[])
            existing_ids = set(existing["ids"])

            new_records = [
                i for i, doc_id in enumerate(ids)
                if doc_id not in existing_ids
            ]

            if not new_records:
                print(
                    "No new documents to add. "
                    "All chunks already exist in ChromaDB."
                )
                print(
                    f"Total documents in collection: "
                    f"{self.collection.count()}"
                )
                return

            self.collection.add(
                ids=[ids[i] for i in new_records],
                embeddings=[embeddings_list[i] for i in new_records],
                metadatas=[metadatas[i] for i in new_records],
                documents=[documents_txt[i] for i in new_records]
            )

            print(f"Added {len(new_records)} new document chunks.")
            print(
                f"Skipped {len(ids) - len(new_records)} existing chunks."
            )
            print(
                f"Total documents in collection: "
                f"{self.collection.count()}"
            )

        except Exception as e:
            print(f"Error adding documents to vector store: {e}")
            raise


vectorstore = Vector_Store()


Vector store initialized. Collection: pdf_documents
Existing document in collection: 671


In [12]:
# Convert chunks to text
texts = [doc.page_content for doc in chunks]

# Generate embeddings
embeddings = embedding_manager.generate_embeddings(texts)

# Store in ChromaDB
vectorstore.add_documents(chunks, embeddings)

Generating embeddings for 61 texts...


Batches: 100%|â–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆ| 2/2 [00:02<00:00,  1.29s/it]


Generated embeddings with shape: (61, 384)
Adding 61 documents must match number of embeddings
Successfully added 61 document to vector store
Total documents in collection: 732


Retrival Pipeline From Vectortore

In [13]:
class RAGRetriever:
    """Handles query-based retrieval from the vector store."""

    def __init__(self, vector_store, embedding_manager):
        """Initialize the retriever."""
        self.vector_store = vector_store
        self.embedding_manager = embedding_manager

    def retrieve(self, query: str, top_k: int = 5, score_threshold=None):
        """Retrieve the most relevant documents from ChromaDB.

        ChromaDB returns distances where smaller values are better.
        The default behavior therefore returns the top-k results without
        applying an incorrect 1-distance similarity filter.
        """

        print(f"Retrieving documents for query: '{query}'")
        print(f"Top K: {top_k}, score threshold: {score_threshold}")

        query_embedding = self.embedding_manager.generate_embeddings([query])[0]

        try:
            results = self.vector_store.collection.query(
                query_embeddings=[query_embedding.tolist()],
                n_results=top_k
            )

            retrieved_docs = []

            if results.get("documents") and results["documents"][0]:
                documents = results["documents"][0]
                metadatas = results["metadatas"][0]
                distances = results["distances"][0]
                ids = results["ids"][0]

                for i, (doc_id, document, metadata, distance) in enumerate(
                    zip(ids, documents, metadatas, distances)
                ):
                    if score_threshold is not None and distance > score_threshold:
                        continue

                    retrieved_docs.append({
                        "id": doc_id,
                        "content": document,
                        "metadata": metadata,
                        "distance": distance,
                        "rank": len(retrieved_docs) + 1
                    })

                print(
                    f"Retrieved {len(retrieved_docs)} documents after filtering"
                )
            else:
                print("No documents found")

            return retrieved_docs

        except Exception as e:
            print(f"Error during retrieval: {e}")
            return []


rag_retriever = RAGRetriever(vectorstore, embedding_manager)
print("RAG retriever initialized successfully")


In [14]:
rag_retriever


In [15]:
rag_retriever.retrieve("Mahesh")


Retriving documents for query: 'Mahesh'
Top K: 5, score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|â–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆ| 1/1 [00:00<00:00, 15.75it/s]

Generated embeddings with shape: (1, 384)
Retrived 5 documents(after filtering)


[{'id': 'doc_33ef65fa_23',
  'content': 'Aim: \nThe primary aim of the internship program at CodSoft was to showcase and improve my machine \nlearning and programming skills through practical projects. During the internship, I focused on \nunderstanding machine learning concepts, working with datasets, and developing predictive models using \nPython. I also reflected on the challenges faced, the solutions implemented, and the improvement in my',
  'metadata': {'modDate': "D:20260430112515+05'30'",
   'moddate': '2026-04-30T11:25:15+05:30',
   'doc_index': 23,
   'source': 'C:\\Users\\Mahesh\\rag_project\\venv\\data\\text_file\\Internship report.pdf',
   'producer': 'MicrosoftÂ® Word 2024',
   'format': 'PDF 1.7',
   'total_pages': 25,
   'page': 10,
   'creationdate': '2026-04-30T11:25:15+05:30',
   'creationDate': "D:20260430112515+05'30'",
   'subject': 'TE IT',
   'title': 'Format for PBS',
   'file_path': 'C:\\Users\\Mahesh\\rag_project\\venv\\data\\text_file\\Internship report.pdf